<a href="https://colab.research.google.com/github/Pramit44/Hospital-los-predictor-Length-of-Stay-/blob/main/Hospital_Length_of__Stay.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib as jb
import json as js

In [ ]:
path=("/content/drive/MyDrive/LengthOfStay.csv")
df=pd.read_csv(path)
df.head()



                              
##                            **EDA**

In [ ]:
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
df["rcount"].value_counts()

In [ ]:
df.hist(figsize=(15,15))


In [ ]:
sns.boxplot(data=df,x="hemo",y="lengthofstay")

In [ ]:
sns.barplot(df,x="respiration",y="lengthofstay")

Changing datatype of dialysisrenalendstage to int
changing datatype of	rcount
drop column

# Data Preprocessing

In [ ]:
df["rcount"]=df["rcount"].str.replace("5+","5").astype(int)

In [ ]:
df.info()

In [ ]:
df=df.drop(["eid","vdate","discharged"],axis=1)

In [ ]:
df

In [ ]:
df["gender"]=df["gender"].map({"M":1,"F":0})

In [ ]:
df.info()

In [ ]:
df = pd.get_dummies(df, columns=["facid"],dtype=int)
df.head()

In [ ]:
df["respiration"].value_counts()

# Feature Engineering

In [ ]:
resp_bins = [0, 5.9, 7.0, 15]
resp_labels = ['Low', 'Normal', 'High']
df['respiration'] = pd.cut(df['respiration'], bins=resp_bins, labels=resp_labels)

In [ ]:
df=pd.get_dummies(df,columns=["respiration"],drop_first=True,dtype=int)


In [ ]:
df["bmi"]=pd.cut(df["bmi"],bins=[0,18.5,24.9,29.9,float("inf")],labels = ['Underweight', 'Normal', 'Overweight', 'Obesity'])

In [ ]:
bins = [0, 6.0, 9.0, 12.0, 16.0, float('inf')]
labels = ['Severe Anemia', 'Moderate Anemia', 'Mild Anemia', 'Normal', 'High (Polycythemia)']
df['hematocrit'] = pd.cut(df['hematocrit'], bins=bins, labels=labels, right=True)

In [ ]:
df['neutrophils'] = pd.cut(df['neutrophils'], bins=[0, 1.5, 7.0, float('inf')], labels=['Low', 'Normal', 'High'])

In [ ]:
df['sodium'] = pd.cut(df['sodium'], bins=[0, 135, 145, float('inf')], labels=['Low', 'Normal', 'High'])

In [ ]:
df['glucose'] = pd.cut(df['glucose'], bins=[0, 100, 125, float('inf')], labels=['Normal', 'Prediabetes', 'Diabetes'])

In [ ]:
df['bloodureanitro'] = pd.cut(df['bloodureanitro'], bins=[0, 7, 20, float('inf')], labels=['Low', 'Normal', 'High'])

In [ ]:
df['creatinine'] = pd.cut(df['creatinine'], bins=[0, 0.5, 1.2, float('inf')], labels=['Low', 'Normal', 'High'])

In [ ]:
cat_cols = ['bmi','hematocrit', 'neutrophils', 'sodium', 'glucose','bloodureanitro', 'creatinine']


In [ ]:
df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)

In [ ]:
reporting_bins = [-1, 4, 7, float('inf')]
reporting_labels = ['Rare / Dormant Category', 'Moderate-Load Routine Case', 'Critical High-Load Condition']

In [ ]:
df['secondarydiagnosisnonicd9'] = pd.cut(
df['secondarydiagnosisnonicd9'],
bins=reporting_bins,
labels=reporting_labels
)

In [ ]:
df=pd.get_dummies(df,columns=["secondarydiagnosisnonicd9"],drop_first=True,dtype=int)

In [ ]:
bins = [0, 1, 2, 3, 4, 5, float('inf')]
labels = [
    'Baseline / No Risk',
    'Low Risk',
    'Moderate Risk',
    'High Risk',
    'High Super-Utilizer',
    'Extreme Super-Utilizer'
]

In [ ]:
df['rcount'] = pd.cut(df['rcount'], bins=bins, labels=labels)

In [ ]:
df=pd.get_dummies(df,columns=["rcount"],drop_first=True,dtype=int)

In [ ]:
display(df.head(3))


In [ ]:
df["pulse"].value_counts()

In [ ]:
pulse_bins = [0, 60, 101, float('inf')]
pulse_labels = ['Bradycardia', 'Normal_Pulse', 'Tachycardia']
df['pulse'] = pd.cut(df['pulse'], bins=pulse_bins, labels=pulse_labels)

In [ ]:
df = pd.get_dummies(df, columns=['pulse'], drop_first=True, dtype=int)

In [ ]:
df.head(3)


In [ ]:
#Changing all the Features into INT DType
df.astype(int)

# MODEL SELECTION

In [ ]:
x=df.drop("lengthofstay",axis=1)
y=df["lengthofstay"]

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

Linear Regression Model

In [ ]:
model=LinearRegression()
model.fit(x_train,y_train)

In [ ]:
y_predict=model.predict(x_test)

In [ ]:
print(f"Mean Squared Error: {mean_squared_error(y_test, y_predict)}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_predict)):.2f} days")
r2 = r2_score(y_test, y_predict)
print(f"R-squared Score: {r2:.4f}")

XGBoost Model

In [ ]:
from xgboost import XGBRegressor
import numpy as np
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
xgb_model.fit(x_train, y_train)


In [ ]:
y_pred_xgb = xgb_model.predict(x_test)
print(f"XGBoost MSE: {mean_squared_error(y_test, y_pred_xgb)}")
print(f"XGBoost RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_xgb)):.2f} days")
print(f"XGBoost R-squared Score: {r2_score(y_test, y_pred_xgb):.4f}")

RandomForestRegressor Model

In [ ]:
from sklearn.ensemble import RandomForestRegressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(x_train, y_train)
y_pred_rf = rf_model.predict(x_test)


In [ ]:
print(f"Random Forestt MSE: {mean_squared_error(y_test, y_pred_rf)}")
print(f"Random Forest RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_rf)):.2f} days")
print(f"Random Forest R-squared Score: {r2_score(y_test, y_pred_rf):.4f}")

GridSearchCV and XGBoost Model

In [ ]:
from sklearn.model_selection import GridSearchCV
xgb_base = XGBRegressor(random_state=42)
param_grid = {'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2]
}
grid_search = GridSearchCV(estimator=xgb_base,
                           param_grid=param_grid,
                           cv=5,
                           scoring='r2',
                           n_jobs=-1,
                           verbose=2)
grid_search.fit(x_train, y_train)
grid_search.fit(x_train, y_train)

Fitting 5 folds for each of 18 candidates, totalling 90 fits
Fitting 5 folds for each of 18 candidates, totalling 90 fits


In [ ]:
best_xgb_model = grid_search.best_estimator_
y_pred_tuned = best_xgb_model.predict(x_test)
mse_tuned = mean_squared_error(y_test, y_pred_tuned)
rmse_tuned = np.sqrt(mse_tuned)
r2_tuned = r2_score(y_test, y_pred_tuned)
print("--- Grid Search Cross-Validation Results ---")
print(f"Optimal Hyperparameters: {grid_search.best_params_}")
print(f"Best CV R-squared Score (Training): {grid_search.best_score_:.4f}\n")

print("--- Tuned XGBoost Model Performance (Test Data) ---")
print(f"Mean Squared Error (MSE)  : {mse_tuned:.4f}")
print(f"Root Mean Sq. Error (RMSE): {rmse_tuned:.2f} days")
print(f"R-squared Score (R2)      : {r2_tuned:.4f}")

## Finalizing the Best Model & Model Building

In [ ]:
import joblib as jb
import json as js

In [ ]:
jb_model=jb.dump(best_xgb_model,"'xgboost_los_model.pkl'")

In [ ]:
model_colms=list(x_train.columns)
with open("model_columns.json","w") as f:
  f.write(js.dumps(model_colms))

In [ ]:
import shutil

# Define the destination path in Google Drive
drive_path = '/content/drive/MyDrive/'

# Copy the XGBoost model file to Google Drive
shutil.copy("'xgboost_los_model.pkl'", drive_path + "xgboost_los_model.pkl")
print("Model 'xgboost_los_model.pkl' saved to Google Drive.")